In [77]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, cast, current_date, current_timestamp, to_date, row_number, date_format, year, sum, desc, spark_partition_id, from_json, to_json
from pyspark.sql.window import Window

In [43]:
spark= (SparkSession
        .builder
        .master("local[*]")
        .appName('CSV reading')
        .getOrCreate()
       )

In [5]:
spark

In [13]:
df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/home/jupyter/data/emp.csv')

In [8]:
!pwd

/home/jupyter/data


In [14]:
df.show()

+-----------+-------------+-------------+---+------+------+-------------------+
|employee_id|department_id|         name|age|gender|salary|          hire_date|
+-----------+-------------+-------------+---+------+------+-------------------+
|          1|          101|     John Doe| 30|  Male| 50000|2015-01-01 00:00:00|
|          2|          101|   Jane Smith| 25|Female| 45000|2016-02-15 00:00:00|
|          3|          102|    Bob Brown| 35|  Male| 55000|2014-05-01 00:00:00|
|          4|          102|    Alice Lee| 28|Female| 48000|2017-09-30 00:00:00|
|          5|          103|    Jack Chan| 40|  Male| 60000|2013-04-01 00:00:00|
|          6|          103|    Jill Wong| 32|Female| 52000|2018-07-01 00:00:00|
|          7|          101|James Johnson| 42|  Male| 70000|2012-03-15 00:00:00|
|          8|          102|     Kate Kim| 29|Female| 51000|2019-10-01 00:00:00|
|          9|          103|      Tom Tan| 33|  Male| 58000|2016-06-01 00:00:00|
|         10|          104|     Lisa Lee

In [27]:
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date, bad_record string"

In [28]:
df_p = spark.read.format('csv').option('header', True).schema(_schema).option('columnNameOfCorruptRecord', 'bad record').load('/home/jupyter/data/emp_new.csv')

In [29]:
df_p.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- bad_record: string (nullable = true)



In [30]:
df_p.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|bad_record|
+-----------+-------------+-------------+---+------+-------+----------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|      null|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|      null|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|      null|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|      null|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|      null|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|      null|
|          7|          101|James Johnson| 42|  Male|   null|2012-03-15|      null|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|      null|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|      null|
|   

In [32]:
df_p.where('bad_record is Null').show()

+-----------+-------------+-------------+---+------+-------+----------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|bad_record|
+-----------+-------------+-------------+---+------+-------+----------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|      null|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|      null|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|      null|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|      null|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|      null|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|      null|
|          7|          101|James Johnson| 42|  Male|   null|2012-03-15|      null|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|      null|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|      null|
|   

In [34]:
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"

df_m = spark.read.format('csv').option('header', True).option('mode', 'DROPMALFORMED').schema(_schema)\
            .load('/home/jupyter/data/emp_new.csv')

In [35]:
df_m.show()

+-----------+-------------+-----------+---+------+-------+----------+
|employee_id|department_id|       name|age|gender| salary| hire_date|
+-----------+-------------+-----------+---+------+-------+----------+
|          1|          101|   John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101| Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|  Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|  Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|  Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|  Jill Wong| 32|Female|52000.0|2018-07-01|
|          8|          102|   Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|    Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|   Lisa Lee| 27|Female|47000.0|2018-08-01|
|         12|          105| Susan Chen| 31|Female|54000.0|2017-02-15|
|         13|          106|  Brian Kim| 45|  Male|75000.0|2011-07-01|
|         14|       

In [37]:
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"

df_f = spark.read.format('csv').option('header', True).option('mode', 'FAILFAST').schema(_schema)\
            .load('/home/jupyter/data/emp_new.csv')

In [38]:
df_f.show()

Py4JJavaError: An error occurred while calling o152.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 12.0 failed 1 times, most recent failure: Lost task 0.0 in stage 12.0 (TID 12) (b199cfacb803 executor driver): org.apache.spark.SparkException: Malformed records are detected in record parsing. Parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.malformedRecordsDetectedInRecordParsingError(QueryExecutionErrors.scala:1399)
	at org.apache.spark.sql.catalyst.util.FailureSafeParser.parse(FailureSafeParser.scala:68)
	at org.apache.spark.sql.catalyst.csv.UnivocityParser$.$anonfun$parseIterator$2(UnivocityParser.scala:421)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apac

In [39]:
_options = {
    'header' : True,
    'inferSchema': True,
    'mode': 'PERMISSIVE'
}

df_opts = (
    spark.read.format('csv').options(**_options).load('/home/jupyter/data/emp_new.csv')
)

In [40]:
df_opts.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|          1|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|          2|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|          3|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|          4|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|          5|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|          6|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|          7|          101|James Johnson| 42|  Male|   Low|2012-03-15|
|          8|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|         10|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|         11|          104|   David Park| 38|  Male| 65000|   no date|
|     

In [45]:
df_parquet = spark.read.format('parquet').load('/home/jupyter/data/sales_data.parquet')

In [47]:
df_parquet.printSchema()

root
 |-- transacted_at: timestamp (nullable = true)
 |-- trx_id: integer (nullable = true)
 |-- retailer_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: integer (nullable = true)



In [51]:
df_parquet.show()

+-------------------+----------+-----------+--------------------+-------+----------+
|      transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+-------------------+----------+-----------+--------------------+-------+----------+
|2017-11-24 19:00:00|1995601912| 2077350195|Walgreen       11-25| 197.23| 216510442|
|2017-11-24 19:00:00|1734117021|  644879053|unkn    ppd id: 7...|   8.58| 930259917|
|2017-11-24 19:00:00|1734117022|  847200066|Wal-Mart  ppd id:...|1737.26|1646415505|
|2017-11-24 19:00:00|1734117030| 1953761884|Home Depot     pp...|  384.5| 287177635|
|2017-11-24 19:00:00|1734117089| 1898522855| Target        11-25|  66.33|1855530529|
|2017-11-24 19:00:00|1734117117|  997626433|Sears  ppd id: 85...| 298.87| 957346984|
|2017-11-24 19:00:00|1734117123| 1953761884|unkn   ppd id: 15...|  19.55|  45522086|
|2017-11-24 19:00:00|1734117152| 1429095612|Ikea     arc id: ...|   9.39|1268541279|
|2017-11-24 19:00:00|1734117153|  847200066|unkn        Kings...|

In [52]:
df_orc = spark.read.format('orc').load('/home/jupyter/data/sales_data.orc')

In [53]:
df_orc.printSchema()

root
 |-- transacted_at: timestamp (nullable = true)
 |-- trx_id: integer (nullable = true)
 |-- retailer_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: integer (nullable = true)



In [54]:
df_singlej = spark.read.format('json').load('/home/jupyter/data/order_singleline.json')

In [55]:
df_singlej.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)



In [57]:
df_multi = spark.read.format('json').option('multiLine', True).load('/home/jupyter/data/order_multiline.json')

In [59]:
df_multi.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)



In [60]:
_schema = 'contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>'

In [62]:
df = spark.read.format('text').load('/home/jupyter/data/order_singleline.json')

In [63]:
df.show()

+--------------------+
|               value|
+--------------------+
|{"order_id":"O101...|
+--------------------+



In [65]:
df_expanded = df.withColumn('parsed', from_json(df.value, _schema))

In [66]:
df_expanded.show()

+--------------------+--------------------+
|               value|              parsed|
+--------------------+--------------------+
|{"order_id":"O101...|{[9000010000, 900...|
+--------------------+--------------------+



In [67]:
df_expanded.printSchema()

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)



In [76]:
df_c = df_expanded.select(col('parsed.order_line_items.amount'))

In [73]:
df_c.show()

+--------------+
|        amount|
+--------------+
|[102.45, 2.01]|
+--------------+



In [81]:
unparsed = df_expanded.withColumn('unparsed', to_json(df_expanded.parsed))

In [84]:
unparsed.show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                              |parsed                                                                      |unparsed                                                                                                                                                                               |
+-----------------------------------------------------------------------------------------------------------------

In [87]:
df_1 = df_expanded.select('parsed.*')

In [88]:
df_1.show()

+--------------------+-----------+--------+--------------------+
|             contact|customer_id|order_id|    order_line_items|
+--------------------+-----------+--------+--------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|
+--------------------+-----------+--------+--------------------+



In [86]:
df_expanded.printSchema()

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)



In [89]:
from pyspark.sql.functions import explode

In [90]:
df_2 = df_1.withColumn('expanded_line_items', explode('order_line_items'))

In [91]:
df_2.show()

+--------------------+-----------+--------+--------------------+-------------------+
|             contact|customer_id|order_id|    order_line_items|expanded_line_items|
+--------------------+-----------+--------+--------------------+-------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|  {102.45, I001, 6}|
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|    {2.01, I003, 2}|
+--------------------+-----------+--------+--------------------+-------------------+



In [92]:
df_3 = df_2.select('contact', 'customer_id', 'order_id', 'expanded_line_items.*')

In [93]:
df_3.show()

+--------------------+-----------+--------+------+-------+---+
|             contact|customer_id|order_id|amount|item_id|qty|
+--------------------+-----------+--------+------+-------+---+
|[9000010000, 9000...|       C001|    O101|102.45|   I001|  6|
|[9000010000, 9000...|       C001|    O101|  2.01|   I003|  2|
+--------------------+-----------+--------+------+-------+---+



In [94]:
df_3 = df_2.withColumn('expand_contact', explode('contact'))

In [98]:
df_3.drop('contact').show()

+-----------+--------+--------------------+-------------------+--------------+
|customer_id|order_id|    order_line_items|expanded_line_items|expand_contact|
+-----------+--------+--------------------+-------------------+--------------+
|       C001|    O101|[{102.45, I001, 6...|  {102.45, I001, 6}|    9000010000|
|       C001|    O101|[{102.45, I001, 6...|  {102.45, I001, 6}|    9000010001|
|       C001|    O101|[{102.45, I001, 6...|    {2.01, I003, 2}|    9000010000|
|       C001|    O101|[{102.45, I001, 6...|    {2.01, I003, 2}|    9000010001|
+-----------+--------+--------------------+-------------------+--------------+



In [99]:
spark.sparkContext.defaultParallelism

8

In [100]:
# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [101]:
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [102]:
emp.rdd.getNumPartitions()

8

In [105]:
emp.write.format('parquet').save('data/Data/empp.parquet')

In [106]:
emp.withColumn('partition_id', spark_partition_id()).show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|           0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|           0|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|           1|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|           1|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|           2|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|           2|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|           3|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|           3|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|      

In [109]:
emp.write.format('csv').option('header', True).save('data/Data/emp2.csv')

In [110]:
emp.write.format('csv').partitionBy('department_id').option('header', True).save('data/Data/emp3.csv')

In [112]:
emp.write.format('csv').mode('overwrite').option('header', True).save('data/Data/emp2.csv')

In [113]:
emp.repartition(1).write.format('csv').option('header', True).save('data/Data/emp4.csv')